# 03 — Analyse

**Cartographier les manques des ERP** · Ameth Yade · M2 SIGLIS, Université de Pau et des Pays de l'Adour

---

> **Qu'est-ce qui manque réellement aux ERP, et où ces manques se concentrent-ils ?**

Trois règles tenues d'un bout à l'autre de ce notebook :

1. **Un chiffre sans interprétation n'est pas une analyse.** Chaque calcul est suivi
   d'une phrase qui dit ce qu'il montre — et ce qu'il ne montre pas.
2. **Ne jamais conclure depuis un seul chiffre.** On croise au moins deux angles.
3. **Chercher activement ce qui contredit l'hypothèse.** C'est le réflexe qui sépare
   un analyste d'un militant.

### Le piège d'interprétation, posé d'emblée

Beaucoup de modules dans une fonction **ne signifie pas** « Odoo est mauvais sur ce
point ». Cela peut signifier :

- un besoin réellement non couvert par le standard,
- un besoin trop spécifique pour être standardisé,
- ou simplement une rubrique où il est facile de publier.

Cette nuance vaut pour tous les résultats qui suivent.

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message=".*OpenSSL.*")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

FIGURES = Path("../figures")
FIGURES.mkdir(exist_ok=True)

df = pd.read_csv("../data/processed/modules_clean.csv")
print(f"{len(df):,} lignes (module x serie) · {df.technical_name.nunique():,} modules distincts")

## Le style des figures

Cinq graphiques lisibles valent mieux que quinze décoratifs. Les règles appliquées :

- **un titre qui énonce le résultat**, pas le sujet (« Les modules comptables dominent »
  plutôt que « Modules par catégorie ») ;
- axes nommés, unités précisées ;
- **la couleur ne sert qu'à distinguer une information**, jamais à décorer ;
- grille et axes discrets, qui ne concurrencent pas les données ;
- **étiquettes directes** sur les séries plutôt qu'une légende à déchiffrer ;
- export en **PDF vectoriel** pour l'inclusion LaTeX.

La palette catégorielle est fixe et ordonnée : une série garde sa couleur quel que soit
le nombre de séries affichées. Elle a été validée pour rester distinguable en cas de
déficience de la vision des couleurs. Trois de ses teintes passent sous le seuil de
contraste sur fond clair — d'où l'étiquetage direct systématique, qui ne laisse jamais
la couleur porter seule l'information.

In [ ]:
import textwrap

# Palette categorielle, ordre fixe. On n'en cycle jamais les teintes : au-dela
# de 6 series, on regroupe en << Autres >> plutot que d'inventer une couleur.
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK = "#0b0b0b"
INK_SOFT = "#52514e"
INK_FAINT = "#b8b7b2"
SURFACE = "#fcfcfb"
ACCENT = "#2a78d6"

mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.size": 9,
    "font.family": "sans-serif",
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.titlepad": 14,
    "axes.labelsize": 9,
    "axes.labelcolor": INK_SOFT,
    "axes.edgecolor": INK_FAINT,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.7,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "pdf.fonttype": 42,   # polices vectorielles, texte selectionnable dans le PDF
})


def finir(ax, titre, sous_titre=None, source=True, largeur=72):
    """Titre qui enonce le resultat, sous-titre explicatif, mention de source.

    Titre et sous-titre sont positionnes en POINTS et non en fraction d'axes :
    une fraction depend de la hauteur de la figure, et les deux blocs finissent
    par se chevaucher des qu'on change `figsize`. Les titres longs sont replies
    plutot que laisses deborder du cadre.
    """
    titre = textwrap.fill(titre, largeur)
    n_lignes_sous = 0
    if sous_titre:
        sous_titre = textwrap.fill(sous_titre, largeur + 8)
        n_lignes_sous = sous_titre.count("\n") + 1
        ax.annotate(sous_titre, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, 7), textcoords="offset points",
                    fontsize=8.5, color=INK_SOFT, va="bottom", ha="left")
    # 7 pt d'ecart + la hauteur du sous-titre + 6 pt de respiration.
    ax.set_title(titre, color=INK, pad=7 + 11 * n_lignes_sous + 6)
    if source:
        ax.figure.text(0.01, -0.02, "Source : Odoo Apps Store, collecte septembre 2026.",
                       fontsize=7, color=INK_FAINT, ha="left")


def enregistrer(fig, nom):
    chemin = FIGURES / f"{nom}.pdf"
    fig.savefig(chemin, bbox_inches="tight")
    fig.savefig(FIGURES / f"{nom}.png", dpi=160, bbox_inches="tight")
    print(f"  -> {chemin.name}")


print("style pret")

## Niveau 1 — Décrire le corpus

### Volumétrie, prix, éditeurs

In [ ]:
n_lignes, n_modules = len(df), df.technical_name.nunique()
n_editeurs = df.author_clean.nunique()

print(f"Lignes (module x serie)   : {n_lignes:,}")
print(f"Modules distincts         : {n_modules:,}")
print(f"Series par module (moy.)  : {n_lignes / n_modules:.2f}")
print(f"Editeurs distincts        : {n_editeurs:,}")
print(f"Modules par editeur (moy.): {n_modules / n_editeurs:.1f}")
print()
print(f"Gratuits : {df.is_free.mean() * 100:.1f} %   Payants : {(~df.is_free).mean() * 100:.1f} %")

payants = df.loc[~df.is_free & df.price_value.notna(), "price_value"]
print(f"\nPrix des modules payants (en euros) :")
print(payants.describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_string())

**Lecture.** Le marché est majoritairement payant, et la distribution des prix est très
étalée : la médiane et la moyenne divergent nettement, signe d'une longue traîne de
modules chers. Ce n'est pas anodin pour la question du projet — un manque que les
éditeurs facturent cher est un manque qu'ils jugent coûteux à combler.

In [ ]:
# Concentration du marche. L'indice de Herfindahl (HHI) est la somme des carres
# des parts de marche, exprime ici en points sur 10 000 comme le font les
# autorites de concurrence. En dessous de 1 500, un marche est dit peu concentre.
parts = df.drop_duplicates("technical_name").author_clean.value_counts(normalize=True)
hhi = (parts ** 2).sum() * 10_000

top = df.drop_duplicates("technical_name").author_clean.value_counts()
for n in (1, 5, 10, 50, 100):
    print(f"  Part des {n:>3} plus gros editeurs : {top.head(n).sum() / top.sum() * 100:5.1f} %")
print(f"\nIndice de Herfindahl (HHI) : {hhi:.0f} / 10 000")
print(f"Editeurs n'ayant publie qu'un seul module : "
      f"{(top == 1).sum():,} ({(top == 1).mean() * 100:.0f} % des editeurs)")

**Lecture.** Le marché est **fragmenté** : un HHI très bas, et une majorité d'éditeurs
n'ayant publié qu'un seul module. C'est cohérent avec la thèse du projet — ces modules
ne sont pas produits par une industrie structurée, mais par des intégrateurs qui
répondent, un par un, à un besoin de client précis. Chaque publication est bien la
trace d'un manque rencontré sur le terrain.

Attention toutefois : quelques gros éditeurs publient massivement, ce qui **gonfle le
comptage** des fonctions où ils sont actifs. On y revient dans les limites.

## Niveau 2 — La carte des manques

### Question 4 — Quelles fonctions métier concentrent le plus de modules tiers ?

C'est la carte elle-même.

In [ ]:
par_fonction = (df.drop_duplicates("technical_name")
                  .fonction.value_counts()
                  .sort_values(ascending=True))
part = par_fonction / par_fonction.sum() * 100

print(f"{'Fonction':<32}{'Modules':>9}{'Part':>8}")
for f, n in par_fonction.sort_values(ascending=False).items():
    print(f"{f:<32}{n:>9,}{part[f]:>7.1f}%")

In [ ]:
# FIGURE 1 -- barres horizontales triees. Une seule serie : pas de legende,
# le titre nomme la grandeur. Etiquettes directes en bout de barre, ce qui
# dispense de lire l'axe et rend la figure lisible en noir et blanc.
fig, ax = plt.subplots(figsize=(7.2, 5.4))

couleurs = [ACCENT if f != "Non renseigne" else INK_FAINT for f in par_fonction.index]
barres = ax.barh(par_fonction.index, par_fonction.values, color=couleurs, height=0.68)

for barre, valeur in zip(barres, par_fonction.values):
    ax.text(valeur + par_fonction.max() * 0.012, barre.get_y() + barre.get_height() / 2,
            f"{valeur:,}".replace(",", " "), va="center", ha="left",
            fontsize=8, color=INK_SOFT)

ax.set_xlabel("Nombre de modules distincts")
ax.set_xlim(0, par_fonction.max() * 1.14)
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, p: f"{int(v):,}".replace(",", " ")))
ax.grid(axis="x", zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)

finir(ax,
      "Le premier besoin des ERP est celui que la taxonomie ne sait pas nommer",
      "Modules tiers distincts par fonction metier, Odoo Apps Store")
enregistrer(fig, "01_modules_par_fonction")
plt.show()

**Lecture.** Le premier bloc n'est pas une fonction métier : c'est la rubrique
**« Outils transverses & technique »**, qui rassemble `Tools`, `Extra Tools` et leurs
voisines. Près de trois modules sur dix relèvent d'un besoin que la taxonomie du site
elle-même ne parvient pas à qualifier.

C'est un résultat, pas un artefact de méthode — et c'était précisément la raison de ne
pas répartir ces rubriques de force dans les autres fonctions.

Viennent ensuite la **comptabilité & finance** et les **ventes & CRM**. Le fait que la
comptabilité arrive en tête des fonctions *nommées* est cohérent avec ce qu'on sait du
domaine : c'est la fonction la plus contrainte par des règles extérieures à l'éditeur.

**Contre-argument à tenir.** Ces trois blocs sont aussi les plus anciens et les plus
larges du périmètre Odoo. Un volume élevé peut simplement refléter une surface
fonctionnelle plus grande, donc plus d'occasions de publier. Le volume seul ne tranche
pas ; c'est l'analyse par version, plus bas, qui apporte l'élément décisif.

### Question 5 — Quelle part des modules répond à une contrainte locale ou réglementaire ?

Cette question mesure le poids du **local dans un produit global**. Un éditeur ne peut
pas couvrir la fiscalité de tous les pays : ce que le standard laisse aux tiers se lit
directement ici.

La détection combine deux signaux dans le titre et le résumé :

- un **vocabulaire réglementaire** (TVA, facturation électronique, normes comptables…),
- une **mention de pays ou de norme nationale**.

C'est une heuristique par mots-clés, avec ses deux erreurs possibles : elle rate les
modules qui n'emploient aucun de ces termes, et elle capte des modules qui les
mentionnent sans être réglementaires. On la mesure plutôt que de la supposer juste.

In [ ]:
TERMES_REGLEMENTAIRES = [
    "vat", "gst", "tva", "tax report", "taxes", "withholding", "fiscal",
    "e-invoic", "einvoic", "electronic invoic", "facturation electronique",
    "compliance", "statutory", "legal report", "localization", "localisation",
    "chart of account", "accounting standard", "ifrs", "gaap", "saf-t", "saft",
    "peppol", "edi", "zatca", "fatoora", "gstr", "sii", "sdi", "ubl",
    "intrastat", "audit file", "tax authority", "tax agency", "declaration",
    "invoice format", "qr code invoice", "fiscal printer", "nomenclature",
]

PAYS = [
    "france", "french", "belgium", "belgian", "germany", "german", "spain",
    "spanish", "italy", "italian", "netherlands", "dutch", "portugal",
    "portuguese", "poland", "polish", "romania", "romanian", "greece", "greek",
    "switzerland", "swiss", "austria", "austrian", "united kingdom", "uk ",
    "ireland", "irish", "denmark", "danish", "sweden", "swedish", "norway",
    "norwegian", "finland", "finnish", "turkey", "turkish", "russia", "russian",
    "ukraine", "india", "indian", "pakistan", "bangladesh", "sri lanka",
    "china", "chinese", "japan", "japanese", "korea", "korean", "vietnam",
    "thailand", "thai", "indonesia", "malaysia", "singapore", "philippines",
    "australia", "australian", "new zealand", "usa", "united states", "u.s.",
    "canada", "canadian", "mexico", "mexican", "brazil", "brazilian",
    "argentina", "chile", "chilean", "colombia", "peru", "ecuador", "uruguay",
    "bolivia", "venezuela", "saudi", "uae", "emirates", "qatar", "kuwait",
    "bahrain", "oman", "egypt", "egyptian", "morocco", "moroccan", "algeria",
    "tunisia", "senegal", "ivory coast", "cote d'ivoire", "ghana", "nigeria",
    "kenya", "tanzania", "uganda", "rwanda", "cameroon", "south africa",
    "ethiopia", "zimbabwe", "zambia", "ohada", "syscohada",
]

texte = (df.title.fillna("") + " " + df.summary_prose.fillna("")).str.lower()
motif_regl = re.compile("|".join(re.escape(t) for t in TERMES_REGLEMENTAIRES))
motif_pays = re.compile("|".join(re.escape(p) for p in PAYS))

df["signal_reglementaire"] = texte.str.contains(motif_regl, regex=True)
df["signal_pays"] = texte.str.contains(motif_pays, regex=True)
df["est_local"] = df.signal_reglementaire | df.signal_pays

uniq = df.drop_duplicates("technical_name")
print(f"Modules portant un signal reglementaire : {uniq.signal_reglementaire.mean() * 100:5.1f} %")
print(f"Modules mentionnant un pays             : {uniq.signal_pays.mean() * 100:5.1f} %")
print(f"Au moins un des deux                    : {uniq.est_local.mean() * 100:5.1f} %")
print(f"\nPour comparaison, la fonction << Localisation & conformite >> "
      f"ne represente que {(uniq.fonction == 'Localisation & conformite').mean() * 100:.1f} % du corpus.")

In [ ]:
# Controle de l'heuristique : on regarde ce qu'elle capte, et ce qu'elle rate.
print("Echantillon de modules DETECTES comme locaux ou reglementaires :\n")
for t in uniq.loc[uniq.est_local, "title"].dropna().sample(8, random_state=0):
    print(f"  + {t[:90]}")

print("\nEchantillon de modules NON detectes, dans la fonction Localisation :\n")
rates = uniq.loc[(uniq.fonction == "Localisation & conformite") & ~uniq.est_local, "title"].dropna()
print(f"  ({len(rates):,} modules rates par l'heuristique dans cette fonction)")
for t in rates.sample(min(6, len(rates)), random_state=0):
    print(f"  - {t[:90]}")

**Lecture.** La contrainte locale ou réglementaire touche une part du corpus **bien
supérieure** à ce que suggère la seule rubrique `Localization` du site. Autrement dit,
le besoin réglementaire ne se range pas dans une case : il traverse la comptabilité, la
facturation, le point de vente et la logistique.

C'est un enseignement direct pour un projet ERP : **la localisation n'est pas un module,
c'est une dimension transverse**, et la sous-estimer est une erreur de chiffrage
classique.

**Ce qui affaiblit ce chiffre.** L'échantillon de contrôle ci-dessus montre les deux
biais à l'œuvre : des modules détectés pour une mention de pays purement commerciale,
et des modules réglementaires ratés faute d'employer le vocabulaire attendu. Le chiffre
est un **ordre de grandeur**, pas une mesure. Il doit être présenté comme tel.

### Question 6 — Quels secteurs d'activité sont visés ?

Un module « pour le BTP » ou « pour la pharmacie » signale un secteur que le standard
sert mal. Deux sources se recoupent ici : la fonction « Verticaux sectoriels » du
référentiel, et la mention explicite d'un secteur dans le texte.

In [ ]:
SECTEURS = {
    "Sante & pharmacie": ["hospital", "clinic", "patient", "pharmacy", "pharmaceutic",
                          "medical", "healthcare", "doctor", "dental", "laboratory"],
    "BTP & immobilier": ["construction", "contractor", "real estate", "property",
                         "building", "architect", "civil work", "site management"],
    "Education": ["school", "university", "student", "education", "academy",
                  "campus", "e-learning", "classroom", "teacher"],
    "Hotellerie & restauration": ["hotel", "restaurant", "booking", "hospitality",
                                  "guest", "reservation", "cafe", "catering"],
    "Agriculture & agroalimentaire": ["farm", "agricultur", "crop", "livestock",
                                      "dairy", "poultry", "food processing", "fishery"],
    "Transport & logistique": ["fleet", "transport", "freight", "trucking",
                               "shipping line", "courier", "last mile", "dispatch"],
    "Industrie & production": ["textile", "garment", "steel", "foundry", "plastic",
                               "chemical", "cement", "furniture manufactur", "printing press"],
    "Commerce de detail": ["retail", "supermarket", "grocery", "boutique",
                           "store chain", "franchise"],
    "Services financiers": ["bank", "insurance", "microfinance", "loan", "lending",
                            "credit union", "leasing"],
    "Associatif & public": ["ngo", "non-profit", "nonprofit", "charity", "municipal",
                            "government", "public sector"],
}

texte_u = (uniq.title.fillna("") + " " + uniq.summary_prose.fillna("")).str.lower()
resultats = {}
for secteur, mots in SECTEURS.items():
    motif = re.compile("|".join(re.escape(m) for m in mots))
    resultats[secteur] = texte_u.str.contains(motif, regex=True).sum()

secteurs = pd.Series(resultats).sort_values(ascending=False)
print(f"{'Secteur':<32}{'Modules':>9}{'Part du corpus':>16}")
for s, n in secteurs.items():
    print(f"{s:<32}{n:>9,}{n / len(uniq) * 100:>15.2f}%")

couverts = texte_u.str.contains(
    "|".join(re.escape(m) for mots in SECTEURS.values() for m in mots), regex=True)
print(f"\nAu moins un secteur identifie : {couverts.mean() * 100:.1f} % du corpus")

**Lecture.** La spécialisation sectorielle reste **minoritaire** : la grande majorité des
modules tiers vise des besoins transverses, pas des métiers particuliers. Le manque
principal des ERP n'est donc pas d'abord sectoriel.

Les secteurs qui ressortent le plus sont ceux dont les processus s'écartent le plus du
modèle générique « achat – stock – vente » sur lequel un ERP est construit.

**Réserve.** Un même module peut compter dans plusieurs secteurs, et les mots-clés
captent aussi des emplois génériques (« store », « bank »). Ces chiffres classent les
secteurs entre eux ; ils ne mesurent pas un volume absolu.

## Niveau 3 — La dynamique, et le résultat principal

### Questions 7 et 8 — Quels manques Odoo a-t-il comblés, lesquels persistent ?

C'est le résultat le plus fort du projet, et celui où l'erreur de méthode guette —
deux fois plutôt qu'une.

**Première erreur : se tromper de source.** La colonne `series` du corpus vient de la
liste non filtrée du store, qui n'affiche chaque module **qu'une fois, à sa version la
plus récente**. Près de 55 % du corpus y apparaît en 19.0, et seuls 260 modules sur
72 316 y figurent sous plus d'une version. Cette colonne mesure la *fraîcheur du
catalogue*, pas sa *couverture par version*. Construire l'analyse temporelle dessus
donnerait une courbe qui monte pour tout le monde, et qui ne dirait rien.

La bonne source est le recensement par `?series=` : il compte les modules
**disponibles** pour chaque version — 277 538 disponibilités, contre 72 579 lignes dans
le corpus.

**Seconde erreur : comparer des effectifs bruts entre versions.** Le catalogue croît
fortement d'une série à l'autre : le marché grandit, et les séries récentes concentrent
l'effort de portage. Une fonction peut doubler en valeur absolue tout en perdant du
terrain.

On raisonne donc en **part de l'offre d'une série donnée**. Si la part d'une fonction
recule, l'éditeur en a absorbé une partie ; si elle tient, le manque est structurel.

Deux précautions supplémentaires :

- les séries anciennes portent des effectifs trop faibles pour que les parts soient
  stables — on les écarte ;
- la série la plus récente est **encore en cours de remplissage** : sa structure n'est
  pas comparable à celle d'une série mature. On l'exclut des tendances.

In [ ]:
SEUIL_SERIE = 500   # en deca, une part n'est pas interpretable

# Le recensement porte sur les 111 categories du site ; on le rabat sur les
# 17 fonctions metier avec la meme table de correspondance qu'au notebook 02.
census = pd.read_csv("../data/interim/census_full.csv")
mapping = pd.read_csv("../data/processed/category_mapping.csv")
census = census.merge(mapping[["category", "fonction"]], on="category", how="left")

manquantes = census.loc[census.fonction.isna(), "category"].unique()
assert len(manquantes) == 0, f"categories non mappees : {manquantes}"

effectifs = census.pivot_table(index="fonction", columns="series",
                               values="n_modules", aggfunc="sum", fill_value=0)
ordre = sorted(effectifs.columns, key=lambda s: float(s))
effectifs = effectifs[ordre]

retenues = [c for c in effectifs.columns if effectifs[c].sum() >= SEUIL_SERIE]
effectifs = effectifs[retenues]
parts = (effectifs / effectifs.sum(axis=0) * 100).round(2)

print(f"Series retenues : {', '.join(retenues)}")
print(f"(exclues : moins de {SEUIL_SERIE} modules disponibles)\n")
print("Modules disponibles par version :")
print(effectifs.sum(axis=0).apply(lambda v: f"{v:,.0f}").to_string())

In [ ]:
# La derniere serie est encore en cours de remplissage : on mesure la tendance
# jusqu'a l'avant-derniere, et on affiche la derniere a part.
debut, fin = retenues[0], retenues[-2]

tendances = pd.DataFrame({
    f"part_{debut}": parts[debut],
    f"part_{fin}": parts[fin],
    f"effectif_{debut}": effectifs[debut],
    f"effectif_{fin}": effectifs[fin],
})
tendances["variation_pts"] = (tendances[f"part_{fin}"] - tendances[f"part_{debut}"]).round(2)
tendances = tendances.sort_values("variation_pts")

print(f"Evolution de la part de l'offre, {debut} -> {fin}\n")
print(tendances.to_string())
print(f"\nSerie {retenues[-1]} exclue du calcul : encore en cours de remplissage.")

**Lecture — le résultat central.**

Les fonctions dont la part **recule** sont celles dont Odoo a absorbé le besoin dans son
standard. Celles dont la part **tient ou progresse** portent un manque que dix versions
successives n'ont pas comblé.

Un piège de lecture, à signaler explicitement dans le document : **une part qui recule
n'est pas un effectif qui recule.** Vérifions-le sur le cas le plus net.

In [ ]:
# Verification du second piege : part en baisse mais effectif en hausse ?
verif = tendances.copy()
verif["part_baisse"] = verif.variation_pts < 0
verif["effectif_hausse"] = verif[f"effectif_{fin}"] > verif[f"effectif_{debut}"]
paradoxe = verif[verif.part_baisse & verif.effectif_hausse]

print("Fonctions dont la PART recule alors que l'EFFECTIF augmente :\n")
if len(paradoxe):
    for f, r in paradoxe.iterrows():
        print(f"  {f:<32} part {r[f'part_{debut}']:5.1f} % -> {r[f'part_{fin}']:5.1f} %   "
              f"effectif {int(r[f'effectif_{debut}']):>6,} -> {int(r[f'effectif_{fin}']):>6,}")
    print("\nPour ces fonctions, Odoo n'a PAS comble le besoin : le reste du")
    print("catalogue a simplement grossi plus vite. Conclure a un manque comble")
    print("serait une erreur d'interpretation.")
else:
    print("  aucune")

In [ ]:
# FIGURE 2 -- l'evolution des parts. Six series au maximum : au-dela, on ne
# cycle pas les couleurs, on regroupe. Etiquettes directes en bout de ligne,
# ce qui evite une legende et garde la figure lisible en noir et blanc.
principales = parts[fin].sort_values(ascending=False).head(6).index.tolist()

fig, ax = plt.subplots(figsize=(7.8, 4.8))
x = np.arange(len(retenues))

for i, fonction in enumerate(principales):
    y = parts.loc[fonction, retenues].values
    ax.plot(x, y, color=SERIES_COLORS[i], marker="o", markersize=3.5,
            markerfacecolor=SURFACE, markeredgewidth=1.4, zorder=3)
    ax.annotate(f" {fonction}", (x[-1], y[-1]), color=SERIES_COLORS[i],
                fontsize=8, va="center", ha="left", fontweight="medium")

# La derniere serie est encore en cours de remplissage : on le montre plutot
# que de laisser croire a une inflexion reelle.
ax.axvspan(len(retenues) - 1.5, len(retenues) - 0.5, color="#f0efeb", zorder=0)
ax.text(len(retenues) - 1, ax.get_ylim()[1] * 0.98, "serie\nen cours",
        fontsize=7, color=INK_FAINT, ha="center", va="top")

ax.set_xticks(x)
ax.set_xticklabels(retenues)
ax.set_xlim(-0.3, len(retenues) + 4.2)
ax.set_xlabel("Version Odoo")
ax.set_ylabel("Part de l'offre de la version (%)")
ax.grid(axis="y", zorder=0)
ax.set_axisbelow(True)

finir(ax,
      "La comptabilite se comble, le point de vente et la productivite se creusent",
      "Part de chaque fonction dans l'offre de modules tiers, par version Odoo")
enregistrer(fig, "02_evolution_par_version")
plt.show()

**Lecture.** Chaque courbe se lit comme une trajectoire de couverture.

Une pente **descendante** signifie qu'Odoo a rattrapé une partie du besoin : les
éditeurs tiers y trouvent moins à faire. Une pente **montante** signale l'inverse — un
besoin qui s'installe plus vite que le standard ne l'absorbe.

Une courbe **plate sur dix versions** est le cas le plus intéressant : ce n'est ni un
oubli ni un retard, c'est un **manque structurel**. Le besoin existe, il est constant,
et l'éditeur a choisi — délibérément ou non — de ne pas le couvrir.

La zone grisée rappelle que la dernière version n'est pas comparable aux autres.

In [ ]:
# FIGURE 3 -- prix par fonction. Echelle logarithmique : les prix s'etendent
# sur trois ordres de grandeur, une echelle lineaire ecraserait tout.
payants_f = df[(~df.is_free) & df.price_value.notna() & (df.price_value > 0)]
ordre_prix = (payants_f.groupby("fonction").price_value.median()
                       .sort_values().index.tolist())
donnees = [payants_f.loc[payants_f.fonction == f, "price_value"].values for f in ordre_prix]

fig, ax = plt.subplots(figsize=(7.4, 5.2))
bp = ax.boxplot(donnees, vert=False, labels=ordre_prix, widths=0.6,
                showfliers=False, patch_artist=True,
                medianprops=dict(color=INK, linewidth=1.6),
                boxprops=dict(facecolor="#d6e4f7", edgecolor=ACCENT, linewidth=1.0),
                whiskerprops=dict(color=INK_FAINT, linewidth=1.0),
                capprops=dict(color=INK_FAINT, linewidth=1.0))

# Part de gratuits en etiquette directe : deux informations, une figure.
for i, f in enumerate(ordre_prix, start=1):
    pct = df.loc[df.fonction == f, "is_free"].mean() * 100
    ax.text(ax.get_xlim()[1], i, f"  {pct:.0f} % gratuits",
            fontsize=7.5, color=INK_SOFT, va="center", ha="left")

ax.set_xscale("log")
ax.set_xlabel("Prix du module payant (euros, echelle logarithmique)")
ax.grid(axis="x", zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)

finir(ax,
      "Les fonctions les plus reglementees sont aussi les plus cheres",
      "Distribution des prix des modules payants et part de gratuits, par fonction")
enregistrer(fig, "03_prix_par_fonction")
plt.show()

**Lecture.** Le prix médian varie fortement d'une fonction à l'autre. Les fonctions où
il est le plus élevé sont celles qui demandent une expertise coûteuse à reproduire —
réglementaire, industrielle, ou métier.

Croisé avec le volume, cela donne une lecture en deux dimensions du manque : un besoin
peut être **fréquent et bon marché** (beaucoup de modules simples) ou **rare et cher**
(peu de modules, mais difficiles). Les deux sont des manques, de nature différente.

**Réserve.** Le prix affiché n'est pas le prix payé : il ne tient compte ni des remises,
ni du coût d'intégration, qui dépasse souvent le prix de la licence.

In [ ]:
# FIGURE 4 -- courbe de Lorenz. Elle montre d'un coup d'oeil si le marche est
# tenu par quelques editeurs ou disperse entre beaucoup.
comptes = np.sort(uniq.author_clean.value_counts().values)
cum_editeurs = np.arange(1, len(comptes) + 1) / len(comptes)
cum_modules = np.cumsum(comptes) / comptes.sum()

# Coefficient de Gini : 0 = parfaitement egalitaire, 1 = un seul editeur.
gini = 1 - 2 * np.trapz(cum_modules, cum_editeurs)

fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.plot([0, 1], [0, 1], color=INK_FAINT, linewidth=1.2, linestyle="--", zorder=2)
ax.plot(cum_editeurs, cum_modules, color=ACCENT, linewidth=2.2, zorder=3)
ax.fill_between(cum_editeurs, cum_modules, cum_editeurs, color="#d6e4f7",
                alpha=0.55, zorder=1)

# Les indices vont en haut a gauche : pour une courbe de Lorenz, cette
# zone est vide par construction, quelle que soit la forme de la courbe.
ax.text(0.04, 0.90, f"Gini = {gini:.2f}", fontsize=11, color=INK, fontweight="bold")
ax.text(0.04, 0.845, f"HHI = {hhi:.0f} / 10 000", fontsize=8.5, color=INK_SOFT)
# Etiquette posee SOUS la diagonale (va="top") et a 45 degres comme elle :
# centree dessus, elle la traversait.
ax.text(0.40, 0.40, "repartition parfaitement egale", fontsize=7.5,
        color=INK_FAINT, rotation=45, rotation_mode="anchor",
        ha="center", va="bottom")

ax.set_xlabel("Part cumulee des editeurs (du plus petit au plus gros)")
ax.set_ylabel("Part cumulee des modules publies")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(zorder=0)
ax.set_axisbelow(True)
ax.set_aspect("equal")

finir(ax,
      "Une poignee d'editeurs publie la majorite des modules",
      "Courbe de Lorenz de la publication de modules par editeur")
enregistrer(fig, "04_concentration_editeurs")
plt.show()

**Lecture.** La courbe s'écarte nettement de la diagonale : une petite minorité
d'éditeurs produit une large part du catalogue, tandis que des milliers d'autres n'ont
publié qu'un module.

Cette double structure est importante pour l'interprétation. Les gros éditeurs publient
des **catalogues**, souvent déclinés sur toutes les versions et toutes les fonctions :
leurs modules reflètent une stratégie commerciale autant qu'un besoin. Les éditeurs à
module unique, eux, sont beaucoup plus proches du signal recherché — **un besoin réel,
rencontré chez un client**.

C'est la limite la plus sérieuse du comptage brut, et elle est traitée ci-dessous.

In [ ]:
# Test de robustesse : la carte des manques tient-elle si l'on retire les
# gros editeurs ? Si le classement change du tout au tout, le resultat
# principal mesure surtout des strategies de catalogue.
gros = uniq.author_clean.value_counts().head(20).index
sans_gros = uniq[~uniq.author_clean.isin(gros)]

comparaison = pd.DataFrame({
    "tous": uniq.fonction.value_counts(normalize=True) * 100,
    "hors_20_plus_gros": sans_gros.fonction.value_counts(normalize=True) * 100,
}).round(1)
comparaison["ecart_pts"] = (comparaison.hors_20_plus_gros - comparaison.tous).round(1)
comparaison = comparaison.sort_values("tous", ascending=False)

print(f"Les 20 plus gros editeurs publient {uniq.author_clean.isin(gros).mean() * 100:.0f} % du corpus.\n")
print(comparaison.to_string())

rang_tous = comparaison.tous.rank(ascending=False)
rang_sans = comparaison.hors_20_plus_gros.rank(ascending=False)
print(f"\nCorrelation des rangs (Spearman) : {rang_tous.corr(rang_sans, method='spearman'):.3f}")

**Lecture.** Si la corrélation des rangs reste élevée, la hiérarchie des fonctions ne
dépend pas des gros éditeurs : la carte des manques est robuste. Les écarts en points
indiquent quelles fonctions sont, elles, portées par des catalogues.

C'est exactement le type de vérification qu'il faut chercher **contre** sa propre
conclusion, et la présenter même — surtout — quand elle la confirme.

In [ ]:
# FIGURE 5 -- les mots des developpeurs eux-memes. On reste simple :
# frequences et bigrammes suffisent, une modelisation thematique n'apporterait
# rien sur des resumes aussi courts.
VIDES = set("""
a an the this that these those and or but if then than with without within for from
to of in on at by is are was were be been being it its your you we our their they he
she as can will would should may might must not no all any each other more most some
such only own same so too very just also into over under about after before between
during through above below up down out off again further once here there when where
why how what which who whom whose module app apps odoo erp new use used using allow
allows allowed enable enables enabled manage managed managing management feature
features functionality system software solution tool tools user users customer
customers create created creating add added adding set setting settings based
different multiple easy easily simple simply automatic automatically option options
available support supports supported help helps helping make makes making get gets
providing provide provides provided view views screen page form list report reports
one two three per via etc ie eg s t
""".split())

MOT_RE = re.compile(r"[a-z][a-z-]{2,}")


def termes(textes, n=12):
    unigrammes, bigrammes = Counter(), Counter()
    for t in textes:
        mots = [m for m in MOT_RE.findall(str(t).lower()) if m not in VIDES]
        unigrammes.update(mots)
        bigrammes.update(f"{a} {b}" for a, b in zip(mots, mots[1:]))
    return unigrammes.most_common(n), bigrammes.most_common(n)


for fonction in parts[fin].sort_values(ascending=False).head(6).index:
    sous = uniq.loc[uniq.fonction == fonction, "summary_prose"].dropna()
    uni, bi = termes(sous, 8)
    print(f"\n{fonction}  ({len(sous):,} modules)")
    print(f"  mots      : {', '.join(m for m, _ in uni)}")
    print(f"  bigrammes : {', '.join(b for b, _ in bi[:5])}")

In [ ]:
# FIGURE 5 -- bigrammes distinctifs. Un terme frequent partout n'apprend rien ;
# on retient ceux qui sont surrepresentes dans une fonction par rapport au
# reste du corpus.
principales_5 = parts[fin].sort_values(ascending=False).head(5).index.tolist()

freq_globale = Counter()
freq_par_fonction = {}
for fonction in principales_5:
    _, bi = termes(uniq.loc[uniq.fonction == fonction, "summary_prose"].dropna(), 4000)
    c = Counter(dict(bi))
    freq_par_fonction[fonction] = c
    freq_globale.update(c)

total_global = sum(freq_globale.values())
distinctifs = {}
for fonction, c in freq_par_fonction.items():
    total_f = sum(c.values())
    scores = {b: (n / total_f) / (freq_globale[b] / total_global)
              for b, n in c.items() if n >= 25}
    distinctifs[fonction] = sorted(scores.items(), key=lambda kv: -kv[1])[:6]

# squeeze=False garantit un tableau 2D meme avec une seule colonne :
# sans lui, subplots renvoie un Axes nu et la boucle echoue des que le
# corpus ne contient qu'une fonction (cas d'une collecte partielle).
fig, axes = plt.subplots(1, len(principales_5), figsize=(2.8 * len(principales_5), 3.6),
                         squeeze=False)
axes = axes.ravel()
for ax, (fonction, items) in zip(axes, distinctifs.items()):
    if not items:
        ax.axis("off")
        continue
    labels = [b for b, _ in items][::-1]
    valeurs = [s for _, s in items][::-1]
    ax.barh(range(len(labels)), valeurs, color=ACCENT, height=0.6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_title(fonction, fontsize=8.5, color=INK)
    ax.set_xlabel("surrepresentation", fontsize=7.5)
    ax.grid(axis="x", zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=7)

fig.suptitle("Les besoins, dans les mots des developpeurs eux-memes",
             fontsize=11, fontweight="bold", x=0.007, ha="left", y=1.06)
fig.text(0.007, 0.99, "Bigrammes les plus surrepresentes dans chaque fonction, "
                      "rapportes a leur frequence dans l'ensemble du corpus",
         fontsize=8.5, color=INK_SOFT, ha="left")
fig.text(0.007, -0.06, "Source : Odoo Apps Store, collecte septembre 2026.",
         fontsize=7, color=INK_FAINT, ha="left")
fig.tight_layout()
enregistrer(fig, "05_termes_par_fonction")
plt.show()

**Lecture.** Ces bigrammes sont la formulation du besoin **par ceux qui l'ont rencontré**.
Là où le comptage par fonction donne une carte à gros grain, ces termes disent
précisément ce qui manquait : un format de document, un mode de calcul, une
intégration avec un service extérieur.

C'est le niveau de détail qui rend l'analyse utile à quelqu'un qui cadre un projet ERP.

## Limites

Cette section est la plus importante du travail. Elle dit ce que l'analyse **ne prouve
pas**.

**Sur la mesure elle-même**

- Le nombre de modules ne mesure pas directement un manque. Une fonction dense peut
  refléter un besoin non couvert, un besoin trop spécifique pour être standardisé, ou
  une rubrique où il est simplement facile de publier.
- Le corpus décrit **l'offre à la date de collecte**, pas l'histoire des publications.
  Un module retiré du store est invisible : les séries anciennes sont donc sous-estimées
  d'une manière qu'on ne peut pas chiffrer.
- La catégorie est **déclarée par l'éditeur**, avec ce que cela suppose d'optimisation
  pour la visibilité.

**Sur les choix de traitement**

- Le regroupement en 17 fonctions est une **interprétation**. Un autre découpage
  donnerait d'autres volumes. La table est versionnée pour que ce choix soit
  contestable.
- La détection du caractère local, réglementaire ou sectoriel repose sur des
  **mots-clés**, avec faux positifs et faux négatifs mesurés plus haut. Ce sont des
  ordres de grandeur.
- La normalisation des éditeurs peut fusionner deux sociétés aux noms proches.

**Sur ce qui n'a pas pu être collecté**

- La note moyenne, la licence et la date de publication ne figurent pas sur les cartes
  de liste.
- Les compteurs d'usage ne sont pas homogènes : téléchargements pour les gratuits,
  achats pour les payants. Ils ne sont jamais additionnés ici.
- Le résidu de 74 modules non attribués à une catégorie (0,08 %) est connu et assumé.

**Sur la portée**

- L'analyse porte sur **un seul écosystème**. Ce qui manque à Odoo ne manque pas
  nécessairement à SAP ou à Dynamics. La méthode, elle, est transposable.

## Synthèse

À reprendre dans l'abstract du document et dans le post, une fois les chiffres
définitifs vérifiés :

1. **Le premier manque est celui qu'on ne sait pas nommer.** La rubrique fourre-tout
   pèse près de trois modules sur dix — plus que n'importe quelle fonction identifiée.
2. **La contrainte locale et réglementaire est transverse**, pas cantonnée à une
   rubrique. Elle touche une part du corpus bien supérieure à ce que la taxonomie du
   site laisse voir.
3. **Certains manques se comblent, d'autres se creusent.** L'analyse par version
   distingue les deux — et montre qu'une part qui recule n'est pas un besoin qui
   disparaît.

Et un résultat de méthode, qui vaut d'être raconté : **la page d'accueil de l'Apps Store
n'expose que 68 % de son propre catalogue.** Le vérifier a changé le périmètre de toute
l'étude.